In [1]:
import numpy as np
import gurobipy as gp
from gurobipy import GRB

In [2]:
model = gp.read("../data/qplib_1493")

model.optimize()

if model.Status != GRB.OPTIMAL:
    raise RuntimeError(
        "Model was not solved optimally"
    )

Set parameter Username
Set parameter LicenseID to value 2829144
Academic license - for non-commercial use only - expires 2027-05-29
Read LP format model from file ../data/qplib_1493.lp
Reading time = 0.00 seconds
obj: 4 rows, 40 columns, 160 nonzeros
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 23.5.0 23F79)

CPU model: Apple M3 Pro
Thread count: 11 physical cores, 11 logical processors, using up to 11 threads

Optimize a model with 4 rows, 40 columns and 160 nonzeros (Min)
Model fingerprint: 0x61c6b1fe
Model has 40 linear objective coefficients
Model has 798 quadratic objective terms
Model has 1 quadratic constraint
Coefficient statistics:
  Matrix range     [1e-02, 1e+00]
  QMatrix range    [1e-02, 2e+00]
  QLMatrix range   [2e-02, 1e+00]
  Objective range  [1e-02, 9e-01]
  QObjective range [2e-02, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e-01, 5e-01]
  QRHS range       [7e+01, 7e+01]


Continuous model is non-convex -- solving as a MIP



In [4]:
vars = model.getVars()

xstar = np.array(
    [v.X for v in vars]
)

objstar = model.ObjVal

print("Optimal objective:")
print(objstar)

Optimal objective:
-43.16043151569994


In [7]:
vars = model.getVars()
n = len(vars)

grad = np.zeros(n)

obj = model.getObjective()

# If the objective is Quadratic
if isinstance(obj, gp.QuadExpr):
    print("Objective is quadratic")
    # Handle quad terms
    for term_idx in range(obj.size()):
        xi = obj.getVar1(term_idx)
        i = xi.index
        val_xi = xi.X

        xj = obj.getVar2(term_idx)
        j = xj.index
        val_xj = xj.X

        coeff = obj.getCoeff(term_idx)

        if i == j:
            # coeff*x_i^2
            grad[i] += 2 * coeff * val_xi
        else:
            # coeff*x_i*x_j
            grad[i] += coeff * val_xj
            grad[j] += coeff * val_xi

        #print(f"Quadratic term: {coeff} * {xi.VarName} * {xj.VarName}")
    
    # Extract the linear portion embedded inside the QuadExpr
    lin_expr = obj.getLinExpr()

else:
    print("Objective is linear")
    lin_expr = obj

# Handle linear terms
for term_idx in range(lin_expr.size()):
    coeff = obj.getCoeff(term_idx)
    i = lin_expr.getVar(term_idx).index
    grad[i] = coeff

    #print(f"Linear term: {coeff} * {lin_expr.getVar(term_idx).VarName}")

Objective is quadratic
Quadratic term: -0.76 * x2 * x2
Quadratic term: -0.3 * x2 * x3
Quadratic term: -0.52 * x2 * x4
Quadratic term: 1.48 * x2 * x5
Quadratic term: 1.92 * x2 * x6
Quadratic term: 1.96 * x2 * x7
Quadratic term: -1.66 * x2 * x8
Quadratic term: -0.58 * x2 * x9
Quadratic term: 1.4 * x2 * x10
Quadratic term: -0.4 * x2 * x11
Quadratic term: 0.32 * x2 * x12
Quadratic term: 0.1 * x2 * x13
Quadratic term: 0.6 * x2 * x14
Quadratic term: 0.32 * x2 * x15
Quadratic term: 0.98 * x2 * x16
Quadratic term: -0.54 * x2 * x17
Quadratic term: 0.8 * x2 * x18
Quadratic term: -0.56 * x2 * x19
Quadratic term: 1.34 * x2 * x21
Quadratic term: 0.96 * x2 * x22
Quadratic term: 0.6 * x2 * x23
Quadratic term: 0.62 * x2 * x24
Quadratic term: -0.2 * x2 * x25
Quadratic term: -0.2 * x2 * x26
Quadratic term: 0.06 * x2 * x27
Quadratic term: 1.16 * x2 * x28
Quadratic term: 1.18 * x2 * x29
Quadratic term: 1.64 * x2 * x30
Quadratic term: -1.7 * x2 * x31
Quadratic term: 1.06 * x2 * x32
Quadratic term: 0.84 * x

In [8]:
norm = np.linalg.norm(grad)

if norm < 1e-12:
    raise RuntimeError(
        "Objective gradient is zero"
    )


direction = -grad / norm

In [10]:
alpha = .01 * np.linalg.norm(xstar)

xnew = xstar + alpha * direction

In [16]:
def getobjectivevalue(gurobimodel, solutionvectordictionary):
    obj = gurobimodel.getObjective()

    if isinstance(obj, gp.QuadExpr):
        objisquad = True
        lin_obj = obj.getLinExpr()
        q_obj = obj - lin_obj
    else:      
        objisquad = False
        lin_obj = obj

    quadterm = 0
    if objisquad:
        for j in range(q_obj.size()):
            v1index = q_obj.getVar1(j).index
            v2index = q_obj.getVar2(j).index
            coeff = q_obj.getCoeff(j)
            thisterm = coeff*solutionvectordictionary[v1index]*solutionvectordictionary[v2index]
            quadterm += thisterm

    linterm = 0
    linterm_noPhi = 0
    for j in range(lin_obj.size()):
        varindex = lin_obj.getVar(j).index
        coeff = lin_obj.getCoeff(j)
        linterm += coeff*solutionvectordictionary[varindex]

    constant = lin_obj.getConstant()
    total = quadterm + linterm + constant

    return total

In [17]:
objnew = getobjectivevalue(model, xnew)


print("\nPerturbed objective:")
print(objnew)

print("\nObjective change:")
print(objnew - objstar)


Perturbed objective:
-42.71894844262914

Objective change:
0.44148307307079904


In [33]:
def evaluate_constraints(model, x):
    """
    Compute the slack of every linear and quadratic constraint in a Gurobi model
    evaluated at a given vector x.

    Parameters
    ----------
    model : gp.Model
        Gurobi model.
    x : numpy array
        Value of every variable, in the same order as model.getVars().

    Returns
    -------
    dict
        Dictionary mapping constraint names to slacks.
    """
    vars = model.getVars()
    if len(x) != len(vars):
        raise ValueError("Length of x must equal number of variables.")

    # Map variable -> value
    #val = {v: float(x[i]) for i, v in enumerate(vars)}

    slacks = {}

    # Linear constraints
    for constr in model.getConstrs():

        row = model.getRow(constr)
        lhs = 0.0
        for term_idx in range(row.size()):
            coeff = row.getCoeff(term_idx)
            i = row.getVar(term_idx).index
            lhs += coeff * x[i]

        rhs = constr.RHS

        if constr.Sense == GRB.LESS_EQUAL:
            slack = rhs - lhs
        elif constr.Sense == GRB.GREATER_EQUAL:
            slack = lhs - rhs
        elif constr.Sense == GRB.EQUAL:
            slack = abs(lhs - rhs)
        else:
            raise RuntimeError(f"Unknown constraint sense {constr.Sense}")
            
        slacks[constr.ConstrName] = slack

    # Quadratic constraints
    for qc in model.getQConstrs():

        qrow = model.getQCRow(qc)

        lhs = 0.0

        # Linear part
        linQc  = qrow.getLinExpr()
        for term_idx in range(linQc.size()):
            coeff = linQc.getCoeff(term_idx)
            xi = x[linQc.getVar(term_idx).index]
            lhs += coeff * xi

        # Quadratic part
        quadQc = qrow - linQc
        for term_idx in range(quadQc.size()):
            i = quadQc.getVar1(term_idx).index
            j = quadQc.getVar2(term_idx).index
            coeff = qrow.getCoeff(i)
            lhs += coeff * x[i] * x[j]

        rhs = qc.QCRHS

        if qc.QCSense == GRB.LESS_EQUAL:
            slack = rhs - lhs
        elif qc.QCSense == GRB.GREATER_EQUAL:
            slack = lhs - rhs
        elif qc.QCSense == GRB.EQUAL:
            slack = abs(lhs - rhs)
        else:
            raise RuntimeError(f"Unknown constraint sense {qc.QCSense}")

        slacks[qc.QCName] = slack

    return slacks

In [34]:
slacks = evaluate_constraints(model, xnew)

print("\nSlacks:")

if len(violations) == 0:
    print("None")

else:
    for name, value in slacks.items():
        print(
            f"{name}: {value}"
        )


Slacks:
e2: 0.0033747132278239833
e3: 0.0019944992115513704
e4: 0.009279818818796581
e5: 0.010309196311353408
e6: 57.69102202490774
